<a href="https://colab.research.google.com/github/jhajagos/SupportingConceptSetGeneration/blob/main/hpo_term_expansion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import pyspark
import json
import logging

In [2]:
UMLS_DIRECTORY = "/content/drive/MyDrive/umls/2025AA/"
UMLS_OUTPUT_DIRECTORY = "/content/drive/MyDrive/umls/2025AA/export/"

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
logging.basicConfig(level=logging.INFO)

In [5]:
spark = pyspark.sql.SparkSession.builder\
    .config("spark.driver.memory", "16g") \
    .getOrCreate()

In [6]:
def attach_catalog_dict(spark_ptr, table_catalog, domains_to_exclude=None):
    """Takes an output dictionary from the OHDSI mapper and maps tables in the domain"""

    db_cat = {}
    if domains_to_exclude is not None:
        filtered_domains = [dn for dn in table_catalog if dn not in domains_to_exclude]
    else:
        filtered_domains = list(table_catalog.keys())

    for domain in filtered_domains:
        for table in table_catalog[domain]:
            location = table_catalog[domain][table]
            if domain not in db_cat:
                db_cat[domain] = {}
            logging.info(f"Loading '{location}' and attaching to namespace as '{table}'")
            db_cat[domain][table] = spark_ptr.read.parquet(location)
            db_cat[domain][table].createOrReplaceTempView(table)

    return db_cat

In [7]:
with open(UMLS_OUTPUT_DIRECTORY + "umls_generated_tables.json", "r") as f:
  catalog_dict = json.load(f)

ct = attach_catalog_dict(spark, catalog_dict)
ct

{'umls': {'MRCONSO': DataFrame[CUI: string, LAT: string, TS: string, LUI: string, STT: string, SUI: string, ISPREF: string, AUI: string, SAUI: bigint, SCUI: string, SDUI: string, SAB: string, TTY: string, CODE: string, STR: string, SRL: int, SUPPRESS: string, CVF: int, dummy: string],
  'MRHIER': DataFrame[CUI: string, AUI: string, CXN: int, PAUI: string, SAB: string, RELA: string, PTR: string, HCD: string, CVF: string, dummy: string],
  'MRDEF': DataFrame[CUI: string, AUI: string, ATUI: string, SATUI: bigint, SAB: string, DEF: string, SUPPRESS: string, CVF: string, dummy: string],
  'MRSAB': DataFrame[VCUI: string, RCUI: string, VSAB: string, RSAB: string, SON: string, SF: string, SVER: string, VSTART: string, VEND: string, IMETA: string, RMETA: string, SLC: string, SCC: string, SRL: int, TFR: int, CFR: int, CXTY: string, TTYL: string, ATNL: string, LAT: string, CENC: string, CURVER: string, SABIN: string, SSN: string, SCIT: string, dummy: string],
  'MRSAT': DataFrame[CUI: string, LU

In [49]:
spark.sql("select distinct LUI,STR from MRCONSO where CUI = 'C0011991' and TTY in ('PN','PT', 'SY') and LAT = 'ENG'").toPandas()

,LUI,STR
0,L0035953,The runs
1,L0811819,diarrhoea symptoms
2,L0318181,Watery stool
3,L0011991,diarrhea
4,L7984957,The trots
5,L1077941,loose bowel motion
6,L1700140,diarrhea running
7,L0011991,diarrheas
8,L0011991,DIARRHEA
9,L1092311,Observation of diarrhea


In [50]:
spark.sql("select * from MRDEF where CUI = 'C0011991' and SAB='HPO'").toPandas()

,CUI,AUI,ATUI,SATUI,SAB,DEF,SUPPRESS,CVF,dummy
0,C0011991,A24673115,AT288958738,NaN,HPO,Abnormally increased frequency (usually define...,N,None,None


#### Function definitions

In [10]:
def expand_term(CUI):
  terms = [t[0] for t in spark.sql(f"select distinct lower(STR) from MRCONSO where CUI = '{CUI}' and TTY in ('PN','PT', 'SY')").toPandas().values.tolist()]
  terms = [t for t in terms if "," not in t and "observation" not in t and "(" not in t and t != "ed"]
  return terms

def find_cui_by_code(code, sab="SNOMEDCT_US"):
  cui_list = [c[0] for c in spark.sql(f"select distinct CUI from MRCONSO where SAB='{sab}' and CODE='{code}' and TTY='PT' order by CUI").toPandas().values.tolist()]
  if len(cui_list):
    return cui_list[0]
  else:
    return None

def find_hpo_label(cui):
  label = spark.sql(f"select STR || ' (' || CODE || ')' from MRCONSO where SAB='HPO' and TTY='PT' and CUI='{cui}'").toPandas().values.tolist()
  if len(label):
    return label[0][0]
  else:
    return None


In [11]:
expand_term("C0015676")

['brain fog', 'mental clouding', 'mental fatigue', 'mental fog']

In [12]:
find_hpo_label("C001d0520")

In [13]:
spark.sql("select STR || ' (' || CODE || ')' from MRCONSO where SAB='HPO' and TTY='PT' and CUI='C0010520'").toPandas().values.tolist()[0][0]

'Cyanosis (HP:0000961)'

In [14]:
hpo_terms_sdf = spark.sql("select * from MRCONSO where where SAB='HPO' and TTY='PT' order by AUI")
hpo_terms_sdf.createOrReplaceTempView("hpo_terms")
hpo_terms_sdf.toPandas()

,CUI,LAT,TS,LUI,STT,SUI,ISPREF,AUI,SAUI,SCUI,SDUI,SAB,TTY,CODE,STR,SRL,SUPPRESS,CVF,dummy
0,C0156273,ENG,P,L0183149,VC,S0819771,N,A24665781,NaN,None,HP:0000015,HPO,PT,HP:0000015,Bladder diverticulum,0,N,256.0,None
1,C1855311,ENG,P,L6456852,PF,S7537070,N,A24665782,NaN,None,HP:0000021,HPO,PT,HP:0000021,Megacystis,0,N,256.0,None
2,C0151721,ENG,S,L0161601,VCW,S0682153,N,A24665784,NaN,None,HP:0000026,HPO,PT,HP:0000026,Male hypogonadism,0,N,256.0,None
3,C0341787,ENG,P,L0525790,PF,S0600488,N,A24665790,NaN,None,HP:0000048,HPO,PT,HP:0000048,Bifid scrotum,0,N,256.0,None
4,C4551492,ENG,P,L0596254,PF,S0688803,N,A24665792,NaN,None,HP:0000054,HPO,PT,HP:0000054,Micropenis,0,N,256.0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19020,C5970443,ENG,P,L19840035,PF,S23637734,Y,A37188032,NaN,None,HP:6001132,HPO,PT,HP:6001132,Elbow lateral collateral ligament tear,0,N,NaN,None
19021,C5970447,ENG,P,L19840032,PF,S23637702,Y,A37188033,NaN,None,HP:6001138,HPO,PT,HP:6001138,Digoxin exposure,0,N,NaN,None
19022,C5970450,ENG,P,L19840051,PF,S23637923,Y,A37188034,NaN,None,HP:6001142,HPO,PT,HP:6001142,Pain at the anterior aspect of the elbow,0,N,NaN,None
19023,C5970455,ENG,P,L19840185,PF,S23637930,Y,A37188036,NaN,None,HP:6001148,HPO,PT,HP:6001148,Pain exacerbated by passive flexion at joint,0,N,NaN,None


In [20]:
hpo_terms_expanded_sdf = spark.sql("""
select distinct hto.*,
hto.STR || ' (' || hto.CODE || ')' as hpo_label,
t.list_terms_expanded, ttt.SNOMED_CODE,
tttt.DEF as hpo_definition, ttttt.term_types
 from (
select mr.CUI, array_agg(distinct lower(mr.STR)) as list_terms_expanded
from MRCONSO mr join hpo_terms ht on mr.CUI = ht.CUI where mr.LAT = 'ENG' group by mr.CUI ) t
join hpo_terms hto on hto.CUI = t.CUI
left outer join (select * from
  (select CUI, AUI, CODE AS SNOMED_CODE, rank() over (partition by CUI order by AUI) as code_rank from MRCONSO
      where SAB='SNOMEDCT_US' and TTY='PT') tt where code_rank = 1) ttt on ttt.CUI = hto.CUI
left outer JOIN (
  select * from MRDEF where SAB = 'HPO'
) tttt on tttt.CUI = hto.CUI
left outer JOIN (select CUI, array_agg(STY) as term_types from MRSTY group by CUI) ttttt on ttttt.CUI = hto.CUI
order by hto.CUI
""")

hpo_terms_expanded_sdf.toPandas()

,CUI,LAT,TS,LUI,STT,SUI,ISPREF,AUI,SAUI,SCUI,...,STR,SRL,SUPPRESS,CVF,dummy,hpo_label,list_terms_expanded,SNOMED_CODE,hpo_definition,term_types
0,C0000727,ENG,P,L0000727,VCW,S0584932,N,A33193765,NaN,None,...,Acute abdomen,0,N,256.0,None,Acute abdomen (HP:0033400),"[abdomen, acute, acute abdomen, acute; abdomen...",9209005,A sudden onset of abdominal pain with associat...,[Sign or Symptom]
1,C0000729,ENG,P,L0000729,VC,S0353650,Y,A30925085,NaN,None,...,Abdominal cramps,0,N,256.0,None,Abdominal cramps (HP:0032155),"[abdominal cramps, cramps abdominal, abdominal...",None,A type of abdominal pain characterized by a fe...,[Sign or Symptom]
2,C0000731,ENG,S,L0000731,PF,S0353653,N,A24677906,NaN,None,...,Abdominal distention,0,N,256.0,None,Abdominal distention (HP:0003270),"[abdomen distended, distended abdomen, abdomin...",60728008,Distention of the abdomen. [https://orcid.org/...,[Finding]
3,C0000734,ENG,P,L0000734,PF,S0582855,N,A28681773,NaN,None,...,Abdominal mass,0,N,256.0,None,Abdominal mass (HP:0031500),"[abdominal mass, mass abdominal, mass;abdomina...",271860004,An abnormal enlargement or swelling in the abd...,[Finding]
4,C0000737,ENG,P,L0000737,VC,S0353662,N,A24670862,NaN,None,...,Abdominal pain,0,N,256.0,None,Abdominal pain (HP:0002027),"[abdominal pain, pain abdominal, pain;abdomina...",21522001,An unpleasant sensation characterized by physi...,[Sign or Symptom]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19034,C5974054,ENG,P,L18793005,PF,S22522080,N,A37187516,NaN,None,...,Abnormally high-pitched voice,0,N,NaN,None,Abnormally high-pitched voice (HP:0001620),"[abnormally high-pitched voice, high pitched v...",None,A persistent (minutes to hours) abnormal incre...,[Finding]
19035,C5975700,ENG,P,L0400543,PF,S0496820,N,A24679618,NaN,None,...,Megalocornea,0,N,256.0,None,Megalocornea (HP:0000485),"[megalocornea, macrocornea, enlarged cornea, a...",None,An enlargement of the cornea with normal clari...,[Anatomical Abnormality]
19036,C5975710,ENG,S,L0576275,PF,S23637813,Y,A37187602,NaN,None,...,Fungal hair infection,0,N,NaN,None,Fungal hair infection (HP:6001088),"[fungal hair infection test positive, fungal h...",None,Demonstration of fungi growing on hair by a me...,[Finding]
19037,C5979869,ENG,P,L18794165,PF,S22523203,N,A35657908,NaN,None,...,Triggered by medication exposure,0,N,256.0,None,Triggered by medication exposure (HP:0034826),[triggered by medication exposure],None,Applies to a sign or symptom that is provoked ...,[Finding]


### Build Hierarchies

In [21]:
from pyspark.sql import functions as F

In [22]:
mrconso_sdf = ct["umls"]["MRCONSO"]
mrhier_sdf = ct["umls"]["MRHIER"]

In [23]:
hpo_pt_sdf = mrconso_sdf.filter((F.col("SAB") == F.lit('HPO')) & (F.col("TTY") == F.lit('PT')))
hpo_pt_sdf = hpo_pt_sdf.withColumn("hpo_label", F.concat(F.col("STR"), F.lit(" ("), F.col("CODE"), F.lit(")")))
hpo_pt_sdf.limit(5).toPandas()

,CUI,LAT,TS,LUI,STT,SUI,ISPREF,AUI,SAUI,SCUI,SDUI,SAB,TTY,CODE,STR,SRL,SUPPRESS,CVF,dummy,hpo_label
0,C2108146,ENG,P,L7134658,PF,S22522743,Y,A35657345,NaN,None,HP:0009626,HPO,PT,HP:0009626,Interphalangeal thumb joint contracture,0,N,256,None,Interphalangeal thumb joint contracture (HP:00...
1,C2108151,ENG,P,L7285964,PF,S22522830,Y,A35657344,NaN,None,HP:0009625,HPO,PT,HP:0009625,Metacarpophalangeal thumb joint contracture,0,N,256,None,Metacarpophalangeal thumb joint contracture (H...
2,C2109272,ENG,P,L7531753,PF,S17128406,Y,A28255893,NaN,None,HP:0025341,HPO,PT,HP:0025341,Corneal keratic precipitates,0,N,256,None,Corneal keratic precipitates (HP:0025341)
3,C2112129,ENG,P,L7248095,PF,S14973290,Y,A24679890,NaN,None,HP:0001830,HPO,PT,HP:0001830,Postaxial foot polydactyly,0,N,256,None,Postaxial foot polydactyly (HP:0001830)
4,C2112942,ENG,P,L7172819,PF,S14973337,Y,A24679897,NaN,None,HP:0001841,HPO,PT,HP:0001841,Preaxial foot polydactyly,0,N,256,None,Preaxial foot polydactyly (HP:0001841)


In [24]:
hpo_mrhier_sdf = mrhier_sdf.filter((F.col("SAB") == F.lit('HPO')) & (F.col("RELA") == F.lit('isa')))

In [25]:
hpo_mrhier_sdf = hpo_mrhier_sdf.withColumn("ptr_list", F.split(F.col("PTR"), r"\."))
hpo_mrhier_sdf.limit(10).toPandas()

,CUI,AUI,CXN,PAUI,SAB,RELA,PTR,HCD,CVF,dummy,ptr_list
0,C4040474,A35658144,1,A35658536,HPO,isa,A24672666.A24681708.A30925678.A35657115.A35658536,None,None,None,"[A24672666, A24681708, A30925678, A35657115, A..."
1,C4040474,A35658144,2,A35658536,HPO,isa,A24672666.A24681708.A30925678.A35657507.A35658536,None,None,None,"[A24672666, A24681708, A30925678, A35657507, A..."
2,C4048199,A24666078,1,A24669545,HPO,isa,A24672666.A24681708.A24674976.A24667854.A24675...,None,None,None,"[A24672666, A24681708, A24674976, A24667854, A..."
3,C4048268,A29930483,1,A24668204,HPO,isa,A24672666.A24681708.A24674976.A24672782.A24670...,None,None,None,"[A24672666, A24681708, A24674976, A24672782, A..."
4,C4048270,A37187945,1,A37187851,HPO,isa,A24672666.A24681708.A24674976.A24668528.A29930...,None,None,None,"[A24672666, A24681708, A24674976, A24668528, A..."
5,C4048270,A37187945,2,A37187851,HPO,isa,A24672666.A24681708.A24674976.A24668682.A24672...,None,None,None,"[A24672666, A24681708, A24674976, A24668682, A..."
6,C4048270,A37187945,3,A37187851,HPO,isa,A24672666.A24681708.A24674976.A24668682.A24672...,None,None,None,"[A24672666, A24681708, A24674976, A24668682, A..."
7,C4048270,A37187945,4,A37187851,HPO,isa,A24672666.A24681708.A24674976.A24670828.A33194...,None,None,None,"[A24672666, A24681708, A24674976, A24670828, A..."
8,C4048270,A37187945,5,A37187851,HPO,isa,A24672666.A24681708.A24674976.A28255534.A29930...,None,None,None,"[A24672666, A24681708, A24674976, A28255534, A..."
9,C4048273,A24675064,1,A26513318,HPO,isa,A24672666.A24681708.A24674976.A24672782.A24672...,None,None,None,"[A24672666, A24681708, A24674976, A24672782, A..."


In [32]:
hpo_mrhier_exploded_sdf = hpo_mrhier_sdf.alias("mh").select(F.col("AUI").alias("child_AUI"),
                                  F.posexplode(F.col("ptr_list"))).filter(F.col("CXN")  == F.lit(1)).join(hpo_pt_sdf.alias("hpo").select("AUI","CUI","CODE","STR","hpo_label"),
                                                                        on=F.col("col")==F.col("hpo.AUI")).distinct()
hpo_mrhier_exploded_sdf.limit(10).toPandas()

,child_AUI,pos,col,AUI,CUI,CODE,STR,hpo_label
0,A29413528,12,A29413696,A29413696,C4072914,HP:0030375,Increased proportion of memory B cells,Increased proportion of memory B cells (HP:003...
1,A37187750,8,A27329520,A27329520,C4280754,HP:0030810,Abnormal tongue physiology,Abnormal tongue physiology (HP:0030810)
2,A32327676,8,A27329520,A27329520,C4280754,HP:0030810,Abnormal tongue physiology,Abnormal tongue physiology (HP:0030810)
3,A24679541,8,A27329520,A27329520,C4280754,HP:0030810,Abnormal tongue physiology,Abnormal tongue physiology (HP:0030810)
4,A24670499,8,A27329520,A27329520,C4280754,HP:0030810,Abnormal tongue physiology,Abnormal tongue physiology (HP:0030810)
5,A33194115,8,A27329520,A27329520,C4280754,HP:0030810,Abnormal tongue physiology,Abnormal tongue physiology (HP:0030810)
6,A37187860,8,A27329520,A27329520,C4280754,HP:0030810,Abnormal tongue physiology,Abnormal tongue physiology (HP:0030810)
7,A37187313,8,A27329520,A27329520,C4280754,HP:0030810,Abnormal tongue physiology,Abnormal tongue physiology (HP:0030810)
8,A24677515,8,A27329520,A27329520,C4280754,HP:0030810,Abnormal tongue physiology,Abnormal tongue physiology (HP:0030810)
9,A32453145,8,A27329520,A27329520,C4280754,HP:0030810,Abnormal tongue physiology,Abnormal tongue physiology (HP:0030810)


In [38]:
hpo_parental_terms_sdf = hpo_mrhier_exploded_sdf.orderBy("child_AUI", "pos").groupBy("child_AUI").agg(F.collect_list("hpo_label").alias("HPO_parental_terms"), F.collect_list("CODE").alias("HPO_parental_codes"))
hpo_parental_terms_sdf.limit(10).toPandas()

,child_AUI,HPO_parental_terms,HPO_parental_codes
0,A24665781,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:0000119, HP:000007..."
1,A24665784,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:0000119, HP:000007..."
2,A24665790,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:0000119, HP:000007..."
3,A24665792,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:0000119, HP:000007..."
4,A24665799,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:0000119, HP:000007..."
5,A24665801,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:0000119]"
6,A24665802,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:0000119, HP:000007..."
7,A24665811,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:0000119, HP:000007..."
8,A24665812,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:0000119, HP:000007..."
9,A24665813,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:0000119, HP:000007..."


In [39]:
hpo_terms_expanded_complete_sdf = hpo_terms_expanded_sdf.alias("htec").join(hpo_parental_terms_sdf.alias("hpt"), F.col("htec.`AUI`") == F.col("hpt.`child_AUI`"), how="left")
hpo_terms_expanded_complete_sdf.count()

19039

In [40]:
hpo_terms_expanded_complete_df = hpo_terms_expanded_complete_sdf.toPandas()


In [43]:
import numpy as np
def convert_to_list(x):
  if type(x) == float:
    return []
  else:
    return x

def reverse_list(x):
  if x is None:
    return []
  else:
    return list(reversed(x))[:-1]

def first_element(x):
  if len(x):
    return x[0]
  else:
    return None

def get_element(x, pos=-2):
  if len(x) >= abs(pos):
    return x[pos]
  else:
    return None

In [44]:
hpo_terms_expanded_complete_df["HPO_parental_terms"] = hpo_terms_expanded_complete_df["HPO_parental_terms"].map(convert_to_list)
hpo_terms_expanded_complete_df["HPO_parental_codes"] = hpo_terms_expanded_complete_df["HPO_parental_codes"].map(convert_to_list)
hpo_terms_expanded_complete_df["HPO_parental_terms_reversed"]  = hpo_terms_expanded_complete_df["HPO_parental_terms"].map(reverse_list)
hpo_terms_expanded_complete_df["HPO_parental_terms_reversed_json"] = hpo_terms_expanded_complete_df["HPO_parental_terms_reversed"].map(lambda x: json.dumps(x))
hpo_terms_expanded_complete_df["HPO_parental_term"] = hpo_terms_expanded_complete_df["HPO_parental_terms_reversed"].map(first_element)
hpo_terms_expanded_complete_df["HPO_second_level_term"] = hpo_terms_expanded_complete_df["HPO_parental_terms_reversed"].map(get_element)
hpo_terms_expanded_complete_df["HPO_parental_codes_json"] = hpo_terms_expanded_complete_df["HPO_parental_codes"].map(lambda x: json.dumps(x))

In [47]:

hpo_terms_expanded_complete_df["list_terms_expanded_json"] = hpo_terms_expanded_complete_df["list_terms_expanded"].map(lambda x: json.dumps(x))
hpo_terms_expanded_complete_df.to_csv(UMLS_OUTPUT_DIRECTORY + "hpo_terms_expanded_complete.csv", index=False)

In [48]:
hpo_terms_expanded_complete_df

,CUI,LAT,TS,LUI,STT,SUI,ISPREF,AUI,SAUI,SCUI,...,term_types,child_AUI,HPO_parental_terms,HPO_parental_codes,HPO_parental_terms_reversed,HPO_parental_terms_reversed_json,HPO_parental_term,HPO_second_level_term,HPO_parental_codes_json,list_terms_expanded_json
0,C0156273,ENG,P,L0183149,VC,S0819771,N,A24665781,NaN,None,...,[Anatomical Abnormality],A24665781,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:0000119, HP:000007...","[Abnormal bladder morphology (HP:0025487), Abn...","[""Abnormal bladder morphology (HP:0025487)"", ""...",Abnormal bladder morphology (HP:0025487),Abnormality of the genitourinary system (HP:00...,"[""HP:0000001"", ""HP:0000118"", ""HP:0000119"", ""HP...","[""bladder diverticulum"", ""diverticulum bladder..."
1,C0151721,ENG,S,L0161601,VCW,S0682153,N,A24665784,NaN,None,...,[Disease or Syndrome],A24665784,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:0000119, HP:000007...","[Hypogonadism (HP:0000135), Abnormality of rep...","[""Hypogonadism (HP:0000135)"", ""Abnormality of ...",Hypogonadism (HP:0000135),Abnormality of the genitourinary system (HP:00...,"[""HP:0000001"", ""HP:0000118"", ""HP:0000119"", ""HP...","[""testicular hypogonadism"", ""hypogonadism; tes..."
2,C0341787,ENG,P,L0525790,PF,S0600488,N,A24665790,NaN,None,...,[Congenital Abnormality],A24665790,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:0000119, HP:000007...","[Abnormal scrotum morphology (HP:0000045), Abn...","[""Abnormal scrotum morphology (HP:0000045)"", ""...",Abnormal scrotum morphology (HP:0000045),Abnormality of the genitourinary system (HP:00...,"[""HP:0000001"", ""HP:0000118"", ""HP:0000119"", ""HP...","[""bifid scrotum"", ""scrotum; bifid"", ""bifid; sc..."
3,C4551492,ENG,P,L0596254,PF,S0688803,N,A24665792,NaN,None,...,[Congenital Abnormality],A24665792,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:0000119, HP:000007...","[Hypoplasia of penis (HP:0008736), Hypoplastic...","[""Hypoplasia of penis (HP:0008736)"", ""Hypoplas...",Hypoplasia of penis (HP:0008736),Abnormality of the genitourinary system (HP:00...,"[""HP:0000001"", ""HP:0000118"", ""HP:0000119"", ""HP...","[""micropenis"", ""small penis"", ""short penis"", ""..."
4,C0042580,ENG,S,L6197661,VC,S0412804,N,A24665799,NaN,None,...,[Disease or Syndrome],A24665799,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:0000119, HP:000007...","[Abnormal ureter physiology (HP:0025634), Abno...","[""Abnormal ureter physiology (HP:0025634)"", ""A...",Abnormal ureter physiology (HP:0025634),Abnormality of the genitourinary system (HP:00...,"[""HP:0000001"", ""HP:0000118"", ""HP:0000119"", ""HP...","[""vesico-ureteral reflux"", ""vesico ureteral re..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19034,C5970421,ENG,P,L19840217,PF,S23637599,Y,A37188026,NaN,None,...,[Finding],A37188026,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:0000598, HP:003170...","[Abnormal vestibular function (HP:0001751), Fu...","[""Abnormal vestibular function (HP:0001751)"", ...",Abnormal vestibular function (HP:0001751),Abnormality of the ear (HP:0000598),"[""HP:0000001"", ""HP:0000118"", ""HP:0000598"", ""HP...","[""abnormal vestibulo-spinal reflex""]"
19035,C5970425,ENG,P,L19840426,PF,S23637934,Y,A37188028,NaN,None,...,[Spatial Concept],A37188028,"[All (HP:0000001), Clinical modifier (HP:00128...","[HP:0000001, HP:0012823, HP:0012830, HP:001283...","[Localized (HP:0012838), Spatial pattern (HP:0...","[""Localized (HP:0012838)"", ""Spatial pattern (H...",Localized (HP:0012838),Position (HP:0012830),"[""HP:0000001"", ""HP:0012823"", ""HP:0012830"", ""HP...","[""palmoplantar location""]"
19036,C5970443,ENG,P,L19840035,PF,S23637734,Y,A37188032,NaN,None,...,[Injury or Poisoning],A37188032,"[All (HP:0000001), Phenotypic abnormality (HP:...","[HP:0000001, HP:0000118, HP:004006